In [1]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset
from datasets import load_dataset
import torch
from transformers import DataCollatorWithPadding

In [2]:
model_name = "bert-base-uncased"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [5]:
print(tokenizer.vocab_size)
print(tokenizer.model_max_length)

30522
512


In [6]:
dataset = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
print(f"Training rows : {dataset['train'].num_rows}")
print(f"Testing rows : {dataset['test'].num_rows}")
print(dataset.column_names)
print(dataset["train"][0]['text'])  # for just text
print(dataset['train'][0])

Training rows : 25000
Testing rows : 25000
{'train': ['text', 'label'], 'test': ['text', 'label'], 'unsupervised': ['text', 'label']}
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM

In [9]:
positive_label_count = dataset['train'].filter(lambda x: x['label']==1)
print(len(positive_label_count))
negative_label = dataset['test'].filter(lambda x: x['label']==0)
print(len(negative_label))
short_reviews = dataset['train'].filter(lambda x: len(x['text'].split()) < 50)
print(len(short_reviews))
long_reviews = dataset['train'].filter(lambda x: len(x['text'].split()) > 50)
print(len(long_reviews))

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

12500


Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

12500


Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

553


Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

24396


In [10]:
# samples
print(dataset['train'][0:10])
print(dataset['train'][30]['label'])
print(dataset['train'][30]['text'])
#

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [11]:
# take one text and tokenize it
single_sample = dataset['train'][0]['text']
single_sample_encoded = tokenizer(single_sample, max_length=512 , padding='max_length' , truncation= True , return_tensors='pt')
print(single_sample_encoded)
print(single_sample_encoded.attention_mask)
print(single_sample_encoded.token_to_chars)


{'input_ids': tensor([[  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
          2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
          2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
          1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
          2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
          6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
          1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
          5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
         14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
          1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
          2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
         25430, 14728,  2245,  2055,  

In [12]:
# concepts of padding
text = [
    "i love to code",
    "i love to train a trasformer model from the scratch"
]
for i in text:
  encoded_text = tokenizer(i , add_special_tokens=True , padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors='pt')


  print(f"Decoded text: {tokenizer.decode(encoded_text['input_ids'][0])}")

Decoded text: [CLS] i love to code [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]

In [13]:
# concept of the truncation
text = [
    "i love to code the model",
    "i love to train a trasformer model from the scratch"
]
for i in text:
  encoded_text = tokenizer(i, add_special_tokens=True, padding="max_length", max_length=5, truncation='only_first' , return_tensors='pt')
  print(f"Original text: '{i}'")
  print(f"Decoded truncated text: {tokenizer.decode(encoded_text['input_ids'][0])}")

Original text: 'i love to code the model'
Decoded truncated text: [CLS] i love to [SEP]
Original text: 'i love to train a trasformer model from the scratch'
Decoded truncated text: [CLS] i love to [SEP]


In [14]:
# when there are batches the the problem occue with the padding then we use the datacollectoe with padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
# column before the tokenization
print(dataset.column_names)

{'train': ['text', 'label'], 'test': ['text', 'label'], 'unsupervised': ['text', 'label']}


In [16]:
# tokineze full dataset
def tokenize_dataset(example):
  return tokenizer(example["text"], truncation=True, padding="max_length", return_tensors='pt', max_length=512)

data_map = dataset.map(tokenize_dataset, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [17]:
print(data_map.column_names)
rmv_text = data_map.remove_columns(['text'])
print(rmv_text.column_names)
renme = data_map.rename_column('label', 'labels')
print(renme)

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'unsupervised': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}
{'train': ['label', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['label', 'input_ids', 'token_type_ids', 'attention_mask'], 'unsupervised': ['label', 'input_ids', 'token_type_ids', 'attention_mask']}
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [18]:
data_map.set_format(type='torch')
print(type(data_map['train'][0]['input_ids']))
print(type(data_map))

<class 'torch.Tensor'>
<class 'datasets.dataset_dict.DatasetDict'>


In [24]:
print(data_map.column_names)
print(data_map.shape)
print(data_map['train']['input_ids'])
print(data_map['train']['attention_mask'])

{'train': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'], 'unsupervised': ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']}
{'train': (25000, 5), 'test': (25000, 5), 'unsupervised': (50000, 5)}
Column([tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
         2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
         2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
         2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
         1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
         2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
         6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
         1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
         5436,  2003,  8857,  2105,  1037,  2402,  4467,  36

In [27]:
# shuffle the dataset
shuffle = data_map.shuffle(seed=42)
small_dataset = data_map['train'].select(range(1000))

In [28]:
print(len(small_dataset))
print(small_dataset)

1000
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})


In [29]:
data = {
    'text': [
        "I finally understood how transformers work after building one from scratch",
        "Financial problems are making my life very difficult right now",
        "My GitHub is growing every single day with new projects",
        "University assignments and freelancing together is very overwhelming",
        "I will get my first Fiverr order very soon",
        "I built a plant disease detector with 99 percent accuracy",
        "I am struggling with money but I never give up",
        "HuggingFace is making my deep learning journey very easy",
        "Sometimes I feel like I cannot handle university and coding together",
        "I am a software engineering student from Lahore Pakistan",
        "Every day I am getting better at machine learning",
        "I do not have enough money to buy proper resources",
        "I completed all six NLP pipelines in just one week",
        "The pressure of financial problems is very stressful",
        "I will earn my first dollar from freelancing very soon"
    ],
    'label': [1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1]
}

In [34]:
df = pd.DataFrame(data)
df

,text,label
0,I finally understood how transformers work aft...,1
1,Financial problems are making my life very dif...,0
2,My GitHub is growing every single day with new...,1
3,University assignments and freelancing togethe...,0
4,I will get my first Fiverr order very soon,1
5,I built a plant disease detector with 99 perce...,1
6,I am struggling with money but I never give up,0
7,HuggingFace is making my deep learning journey...,1
8,Sometimes I feel like I cannot handle universi...,0
9,I am a software engineering student from Lahor...,1


In [37]:
custom_dataset = Dataset.from_pandas(df)

In [39]:
data_collector = DataCollatorWithPadding(tokenizer = tokenizer)

In [41]:
custom_map = custom_dataset.map(tokenize_dataset, batched=True)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]